# BUILDING INCREMENTAL CUSTOMER DIMENSION TABLE


In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable
from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType, IntegerType, DateType
from pyspark.sql.window import Window
import logging

##Logging configuration

In [0]:

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger("silver_crm_cust_info")
logger.setLevel(logging.INFO)

In [0]:

ALLOWED_PREFIX = "AW000"
EXPECTED_LENGTH = 10

BRONZE_TABLE = "workspace.bronze.crm_cust_info"
SILVER_TABLE = "workspace.silver.crm_customers"
QUARANTINE_TABLE = "workspace.quarantine.crm_cust_info"

RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "create_date",
}

# Data Cleaning Utilities
What we do here:
- Rename columns to consistent names
- Fix data types (customer_id → integer, create_date → date)
- Standardize messy categorical values
(gender: M/male/1 → "Male", F/female/2 → "Female")
(marital_status: S/single → "Single", M/married → "Married")
## Why:
Make raw data clean, consistent and ready for analysis / modeling
Prevent errors caused by wrong types or different spellings

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType, IntegerType, DateType


# Remove leading and trailing spaces from all string columns
def trim_string_columns(df):
    for field in df.schema.fields:
        if isinstance(field.dataType, StringType):
            df = df.withColumn(field.name, trim(col(field.name)))
    return df


# Rename columns using provided mapping dictionary
def rename_columns(df, rename_map):
    for old_name, new_name in rename_map.items():
        if old_name in df.columns:
            df = df.withColumnRenamed(old_name, new_name)
    return df


# Cast selected columns to correct data types
def cast_columns(df):
    return (
        df
        .withColumn("customer_id", col("customer_id").cast(IntegerType()))
        .withColumn("create_date", col("create_date").cast(DateType()))
    )


# Standardize categorical values (gender and marital_status)
def standardize_categorical(df):
    return (
        df
        .withColumn(
            "marital_status",
            F.when(F.upper(F.col("marital_status")).isin("S", "SINGLE"), "Single")
             .when(F.upper(F.col("marital_status")).isin("M", "MARRIED"), "Married")
             .otherwise(None)
        )
        .withColumn(
            "gender",
            F.when(F.upper(F.col("gender")).isin("M", "MALE", "1"), "Male")
             .when(F.upper(F.col("gender")).isin("F", "FEMALE", "2"), "Female")
             .otherwise(None)
        )
    )

# Silver-layer data quality validations
##What we do here:
- Check business key is not null or empty → quarantine bad rows
- Check business key has correct format (prefix + length + only numbers)
- Keep best row when duplicate keys exist (most complete + newest)
- Final check: make sure no duplicates or null keys remain before save
##Why:
- Guarantee clean, unique, reliable data in Silver layer
- Catch and quarantine bad data early
- Prevent broken analytics / models later

In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window


# Validate business key is not NULL or empty
def validate_not_null_business_key(df, key_col, quarantine_table):

    invalid_condition = (
        F.col(key_col).isNull() |
        (F.col(key_col) == "")
    )

    df_invalid = df.filter(invalid_condition)
    df_valid = df.filter(~invalid_condition)

    if df_invalid.limit(1).count() > 0:

        df_invalid = (
            df_invalid
            .withColumn("rejection_reason", F.lit("null_or_empty_business_key"))
            .withColumn("rejection_stage", F.lit("silver_null_validation"))
            .withColumn("processed_timestamp", F.current_timestamp())
        )

        delta_table = DeltaTable.forName(df.sparkSession, quarantine_table)

        (
            delta_table.alias("target")
            .merge(
                df_invalid.alias("source"),
                f"""
                (
                target.{key_col} = source.{key_col}
                OR (target.{key_col} IS NULL AND source.{key_col} IS NULL)
                )
                AND target.rejection_stage = source.rejection_stage
                """
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

    return df_valid


# Validate business key format (prefix + length + numeric suffix)
def validate_business_key_format(
    df,
    key_col,
    allowed_prefix,
    expected_length,
    quarantine_table
):

    prefix_length = len(allowed_prefix)

    prefix_match = F.col(key_col).startswith(allowed_prefix)

    length_match = F.length(F.col(key_col)) == expected_length

    numeric_part = F.col(key_col).substr(
        prefix_length + 1,
        expected_length - prefix_length
    )

    numeric_match = numeric_part.rlike("^[0-9]+$")

    valid_condition = prefix_match & length_match & numeric_match

    df_valid = df.filter(valid_condition)
    df_invalid = df.filter(~valid_condition)

    if df_invalid.limit(1).count() > 0:

        df_invalid = (
            df_invalid
            .withColumn("rejection_reason", F.lit("invalid_business_key_format"))
            .withColumn("rejection_stage", F.lit("silver_format_validation"))
            .withColumn("processed_timestamp", F.current_timestamp())
        )

        delta_table = DeltaTable.forName(df.sparkSession, quarantine_table)

        (
            delta_table.alias("target")
            .merge(
                df_invalid.alias("source"),
                f"""
                target.{key_col} = source.{key_col}
                AND target.rejection_stage = source.rejection_stage
                """
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

    return df_valid


# Resolve duplicate business keys
def resolve_duplicate_business_keys(
    df,
    key_col,
    important_cols,
    quarantine_table
):

    df = df.withColumn(
        "completeness_score",
        sum(F.col(c).isNotNull().cast("int") for c in important_cols)
    )

    w = Window.partitionBy(key_col).orderBy(
        F.col("completeness_score").desc(),
        F.col("ingest_timestamp").desc_nulls_last(),
        F.col("customer_id").desc_nulls_last()
    )

    df_ranked = df.withColumn("rn", F.row_number().over(w))

    df_winners = df_ranked.filter(F.col("rn") == 1)
    df_duplicates = df_ranked.filter(F.col("rn") > 1)

    if df_duplicates.limit(1).count() > 0:

        df_duplicates = (
            df_duplicates
            .drop("completeness_score", "rn")
            .withColumn("rejection_reason", F.lit("duplicate_business_key"))
            .withColumn("rejection_stage", F.lit("silver_duplicate_resolution"))
            .withColumn("processed_timestamp", F.current_timestamp())
        )

        delta_table = DeltaTable.forName(df.sparkSession, quarantine_table)

        (
            delta_table.alias("target")
            .merge(
                df_duplicates.alias("source"),
                f"""
                target.{key_col} = source.{key_col}
                AND target.rejection_stage = source.rejection_stage
                """
            )
            .whenNotMatchedInsertAll()
            .execute()
        )

    return df_winners.drop("completeness_score", "rn")


# Final defensive validation before writing to Silver
def final_business_key_validation(df, key_col):

    duplicate_count = (
        df.groupBy(key_col)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    if duplicate_count > 0:
        raise Exception("Silver DQ FAILED: Duplicate keys remain")

    null_count = df.filter(F.col(key_col).isNull()).count()

    if null_count > 0:
        raise Exception("Silver DQ FAILED: Null keys remain")

    return df

# Silver / Gold layer write helpers
##What we do here:
- Remove technical columns we don't want to keep (ingest time, source etc.)
- Full overwrite: replace entire table with new data + allow schema changes
- Incremental merge: upsert records using business key
→ update if key exists, insert if new
→ create table if it doesn't exist yet
##Why:
- Keep clean final tables without extra metadata
- Support both full refresh and daily delta loads
- Safe & efficient updates without duplicates

In [0]:
from delta.tables import DeltaTable


# Drop technical metadata columns
def drop_metadata_columns(df, metadata_cols):
    return df.drop(*metadata_cols)

# Full load write (overwrite)
def write_full_overwrite(df, target_table):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )
    
# Incremental write using MERGE (upsert logic)
def write_incremental_merge(
    df,
    target_table,
    business_key
):
    spark = df.sparkSession
    # If table does not exist → create it
    if not spark.catalog.tableExists(target_table):
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )
        return
    delta_table = DeltaTable.forName(spark, target_table)
    (
        delta_table.alias("target")
        .merge(
            df.alias("source"),
            f"target.{business_key} = source.{business_key}"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )